$\Huge\text{Western US Power Grid}$

$\large\text{Jorge Rodríguez Toledo}$

### Utility functions to perform the full analysis

In [2]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import scipy.stats as stats
from scipy.optimize import curve_fit
import itertools
import community as community_louvain
import warnings
warnings.filterwarnings('ignore')


# SECTION 1: LOAD DATA
# Load network from US power grid file.
def load_network(filepath='USpowergrid.txt'):
    print("SECTION 1: Loading Network Data")
    print("-"*70)

    G = nx.read_edgelist(filepath, nodetype=int, data=(('weight', float),))

    print(f"Successfully loaded network")
    print()

    return G


# SECTION 2: BASIC DESCRIPTORS
# Compute basic network descriptors.
def analyse_basic_descriptors(G):
    print("SECTION 2: Basic Descriptors")
    print("-"*70)

    # Basic properties
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    density = nx.density(G)

    # Connectivity
    is_connected = nx.is_connected(G)
    n_components = nx.number_connected_components(G)

    # Largest connected component
    if not is_connected:
        components = list(nx.connected_components(G))
        largest_cc = G.subgraph(max(components, key=len)).copy()
        gcc_size = largest_cc.number_of_nodes()
        gcc_edges = largest_cc.number_of_edges()
    else:
        largest_cc = G
        gcc_size = n_nodes
        gcc_edges = n_edges

    # Print results
    print(f"Network Size:")
    print(f"  Number of nodes: {n_nodes}")
    print(f"  Number of edges: {n_edges}")
    print(f"  Network density: {density:.6f}")
    print()
    print(f"Connectivity:")
    print(f"  Is connected: {is_connected}")
    print(f"  Number of components: {n_components}")
    print(f"  Largest component: {gcc_size} nodes ({gcc_size/n_nodes*100:.2f}%)")
    print()

    # Save to CSV
    basic_stats = pd.DataFrame({
        'Metric': ['Nodes', 'Edges', 'Density', 'Connected', 
                   'Components', 'LCC_Size', 'LCC_Edges'],
        'Value': [n_nodes, n_edges, density, is_connected, 
                  n_components, gcc_size, gcc_edges]
    })
    basic_stats.to_csv('basic_descriptors.csv', index=False)

    return {
        'n_nodes': n_nodes,
        'n_edges': n_edges,
        'density': density,
        'is_connected': is_connected,
        'largest_cc': largest_cc,
    }


# SECTION 3: CLUSTERING
# Analyse clustering properties of the network.
def analyse_clustering(G):
    print("SECTION 3: Clustering Analysis")
    print("-"*70)

    # Global clustering coefficient
    global_clustering = nx.transitivity(G)

    # Average clustering coefficient
    avg_clustering = nx.average_clustering(G)

    # Local clustering statistics
    local_clustering = nx.clustering(G)
    clustering_values = list(local_clustering.values())
    clustering_mean = np.mean(clustering_values)
    clustering_median = np.median(clustering_values)
    clustering_std = np.std(clustering_values)

    print(f"Clustering Coefficients:")
    print(f"  Global clustering (transitivity): {global_clustering:.6f}")
    print(f"  Average clustering: {avg_clustering:.6f}")
    print(f"  Local clustering mean: {clustering_mean:.6f}")
    print(f"  Local clustering median: {clustering_median:.6f}")
    print(f"  Local clustering std: {clustering_std:.6f}")
    print()

    # Save results
    clustering_df = pd.DataFrame({
        'Node': list(local_clustering.keys()),
        'Clustering': list(local_clustering.values())
    })
    clustering_df.to_csv('local_clustering.csv', index=False)

    clustering_summary = pd.DataFrame({
        'Metric': ['Global_Clustering', 'Average_Clustering', 'Mean_Local', 'Median_Local', 'Std_Local'],
        'Value': [global_clustering, avg_clustering, clustering_mean, clustering_median, clustering_std]
    })
    clustering_summary.to_csv('clustering_summary.csv', index=False)

    # Plot clustering distribution
    fig, ax = plt.subplots(figsize=(12, 7))
    n, bins, patches = ax.hist(clustering_values, bins=50, range=(0, 1), 
                            edgecolor='black', alpha=0.7, color='steelblue', label='Frequency')

    μ = clustering_mean
    σ = clustering_std

    # Std Dev relevant area
    ax.axvspan(max(0, μ-σ), min(1, μ+σ), 
            alpha=0.25, color='orange', label='Mean ± Std. Dev.')
    if (μ - σ) > 0:
        ax.axvline(μ - σ, color='orange', linestyle='--', linewidth=2.0)
    ax.axvline(μ + σ, color='orange', linestyle='--', linewidth=2.0)

    # Mean
    ax.axvline(μ, color='red', linestyle='-', linewidth=2.5, label=f'Mean: {μ:.4f}')

    # Median
    ax.axvline(clustering_median, color='purple', linestyle='-', linewidth=2.5, 
            label=f'Median: {clustering_median:.4f}')

    ax.set_xlabel('Clustering Coefficient', fontsize=24, family="serif")
    ax.set_ylabel('Frequency', fontsize=24, family="serif")
    ax.set_title('Distribution of Local Clustering Coefficients', fontsize=30, family="serif")
    ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
    ax.legend(loc='upper right', framealpha=0.95, prop={'family': 'serif', 'size': 20})
    ax.set_xlim(0, 1)
    ax.tick_params(axis='both', which='major', labelsize=20)

    plt.tight_layout()
    plt.savefig('clustering_distribution.pdf', dpi=300, bbox_inches='tight')
    plt.close()


# SECTION 4: DISTANCES
# Analyse distance metrics in the network.
def analyse_distances(G, largest_cc):
    print("SECTION 4: Distance Analysis")
    print("-"*70)

    # Use largest connected component
    G_analysis = largest_cc

    # Average shortest path length
    avg_shortest_path = nx.average_shortest_path_length(G_analysis)

    # Diameter and radius
    diameter = nx.diameter(G_analysis)
    radius = nx.radius(G_analysis)

    # Eccentricity
    eccentricity = nx.eccentricity(G_analysis)
    ecc_values = list(eccentricity.values())
    avg_eccentricity = np.mean(ecc_values)

    print(f"Distance Metrics:")
    print(f"  Average shortest path length: {avg_shortest_path:.6f}")
    print(f"  Network diameter: {diameter}")
    print(f"  Network radius: {radius}")
    print(f"  Average eccentricity: {avg_eccentricity:.6f}")
    print()

    # Sample path lengths for large networks
    if G_analysis.number_of_nodes() > 1000:
        print("  Sampling path lengths (network is large)...")
        sample_size = min(1000, G_analysis.number_of_nodes())
        sample_nodes = np.random.choice(list(G_analysis.nodes()), 
                                         size=sample_size, replace=False)
        path_lengths = []
        for i in range(len(sample_nodes)):
            for j in range(i+1, len(sample_nodes)):
                try:
                    p = nx.shortest_path_length(G_analysis, sample_nodes[i], sample_nodes[j])
                    path_lengths.append(p)
                except nx.NetworkXNoPath:
                    pass
    else:
        print("  Computing all path lengths...")
        path_lengths = []
        for source in G_analysis.nodes():
            lengths = nx.single_source_shortest_path_length(G_analysis, source)
            path_lengths.extend([l for t, l in lengths.items() if t != source])

    # Save results
    distance_stats = pd.DataFrame({
        'Metric': ['Avg_Shortest_Path', 'Diameter', 'Radius', 'Avg_Eccentricity'],
        'Value': [avg_shortest_path, diameter, radius, avg_eccentricity]
    })
    distance_stats.to_csv('distance_metrics.csv', index=False)

    # Plot path length distribution
    plt.figure(figsize=(10, 6))
    plt.hist(path_lengths, bins=range(1, max(path_lengths)+2), edgecolor='black', alpha=0.7, color='coral')
    plt.xlabel('Path Length', fontsize=24, family="serif")
    plt.ylabel('Frequency', fontsize=24, family="serif")
    plt.title('Distribution of Shortest Path Lengths', fontsize=30, family="serif")
    plt.tick_params(axis='both', which='major', labelsize=20)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('path_length_distribution.pdf', dpi=300, bbox_inches='tight')
    plt.close()


# SECTION 5: DEGREE STATISTICS
# Analyse degree distribution and statistics.
def analyse_degree_statistics(G):
    print("SECTION 5: Degree Statistics")
    print("-"*70)

    # Degree statistics
    degrees = [G.degree(n) for n in G.nodes()]
    avg_degree = np.mean(degrees)
    median_degree = np.median(degrees)
    std_degree = np.std(degrees)
    min_degree = np.min(degrees)
    max_degree = np.max(degrees)

    print(f"Degree Statistics:")
    print(f"  Average degree: {avg_degree:.4f}")
    print(f"  Median degree: {median_degree:.4f}")
    print(f"  Std deviation: {std_degree:.4f}")
    print(f"  Min degree: {min_degree}")
    print(f"  Max degree: {max_degree}")
    print()

    # Save to CSV
    basic_stats = pd.DataFrame({
        'Metric': ['Avg_Degree', 'Median_Degree', 'Std_Degree', 'Min_Degree', 'Max_Degree'],
        'Value': [avg_degree, median_degree, std_degree, min_degree, max_degree]
    })
    basic_stats.to_csv('degree_statistics.csv', index=False)

    # Degree sequence
    degree_sequence = sorted([d for n, d in G.degree()], reverse=True)
    degree_count = Counter(degree_sequence)

    # Degree distribution (PDF)
    degrees_unique = sorted(degree_count.keys())
    freq = np.array([degree_count[k] for k in degrees_unique])
    prob = freq / freq.sum()
    print("Degree Distribution:")

    degree_dist_df = pd.DataFrame({
        'Degree': degrees_unique,
        'Frequency': freq,
        'Probability': prob
    })
    degree_dist_df.to_csv('degree_distribution.csv', index=False)

    # R2 function 
    def r2(y_true, y_pred):
        ss_res = np.sum((y_true-y_pred)**2)
        ss_tot = np.sum((y_true-np.mean(y_true))**2)
        return 1-ss_res/ss_tot if ss_tot>0 else np.nan

    # PDF Fitting
    print("  PDF Fitting:")
    k_fit_pdf = np.array([k for k, p in zip(degrees_unique, prob) if p > 0 and k > 0])
    p_fit_pdf = np.array([p for k, p in zip(degrees_unique, prob) if p > 0 and k > 0])

    pdf_fits = {}  # store curves for plotting

    if len(k_fit_pdf)>5:
        # Power-law PDF
        def power_law_pdf(k, C, gamma):
            return C*k**(-gamma)

        popt_pl_pdf, _ = curve_fit(power_law_pdf, k_fit_pdf, p_fit_pdf,
                                   p0=[1.0, 2.0], maxfev=20000)
        C_pl_pdf, gamma_pl_pdf = popt_pl_pdf
        p_pred_pl_pdf = power_law_pdf(k_fit_pdf, *popt_pl_pdf)
        r2_pl_pdf = r2(p_fit_pdf, p_pred_pl_pdf)
        print(f"    Power-law: P(k) = {C_pl_pdf:.4e} * k^(-{gamma_pl_pdf:.4f}), "
              f"R^2 = {r2_pl_pdf:.4f}")
        pdf_fits['Power-law'] = (power_law_pdf, popt_pl_pdf)

        # Exponential PDF
        def exp_pdf(k, C, λ):
            return C*np.exp(-λ*k)

        popt_exp_pdf, _ = curve_fit(exp_pdf, k_fit_pdf, p_fit_pdf,
                                    p0=[1.0, 0.1], maxfev=20000)
        C_exp_pdf, λ_exp_pdf = popt_exp_pdf
        p_pred_exp_pdf = exp_pdf(k_fit_pdf, *popt_exp_pdf)
        r2_exp_pdf = r2(p_fit_pdf, p_pred_exp_pdf)
        print(f"    Exponential: P(k) = {C_exp_pdf:.4e} * exp(-{λ_exp_pdf:.4f} * k), "
              f"R^2 = {r2_exp_pdf:.4f}")
        pdf_fits['Exponential'] = (exp_pdf, popt_exp_pdf)

        # Log-normal PDF
        def lognormal_pdf(k, A, mu, sigma):
            return A*(1.0/(k*sigma*np.sqrt(2*np.pi)))*np.exp(-(np.log(k)-mu)**2/(2*sigma**2))

        logk = np.log(k_fit_pdf)
        p0_logn = [max(p_fit_pdf), np.mean(logk), np.std(logk)]
        popt_logn_pdf, _ = curve_fit(lognormal_pdf, k_fit_pdf, p_fit_pdf,
                                     p0=p0_logn, maxfev=30000)
        A_logn_pdf, mu_logn_pdf, sigma_logn_pdf = popt_logn_pdf
        p_pred_logn_pdf = lognormal_pdf(k_fit_pdf, *popt_logn_pdf)
        r2_logn_pdf = r2(p_fit_pdf, p_pred_logn_pdf)
        print(f"    Log-normal: A={A_logn_pdf:.4e}, mu={mu_logn_pdf:.4f}, "
              f"sigma={sigma_logn_pdf:.4f}, R^2 = {r2_logn_pdf:.4f}")
        pdf_fits['Log-normal'] = (lognormal_pdf, popt_logn_pdf)

    # CCDF Fitting
    print()
    print("  CCDF fitting:")
    total_nodes = freq.sum()
    ccdf = []
    for k in degrees_unique:
        count_ge_k = sum(c for dk, c in degree_count.items() if dk >= k)
        ccdf.append(count_ge_k / total_nodes)
    ccdf = np.array(ccdf)

    ccdf_df = pd.DataFrame({"Degree": degrees_unique, "CCDF": ccdf})
    ccdf_df.to_csv("degree_ccdf.csv", index=False)   

    ccdf_fits = {}

    try:
        k_fit = np.array([k for k, F in zip(degrees_unique, ccdf) if F > 0 and k > 0])
        F_fit = np.array([F for k, F in zip(degrees_unique, ccdf) if F > 0 and k > 0])

        if len(k_fit)>5:
            logF = np.log10(F_fit)

            def r2_log(y_true_log, y_pred_log):
                ss_res = np.sum((y_true_log-y_pred_log)**2)
                ss_tot = np.sum((y_true_log-np.mean(y_true_log))**2)
                return 1-ss_res/ss_tot if ss_tot>0 else np.nan

            # Power-law CCDF
            def pl_ccdf(k, C, alpha):
                return C*k**(-alpha)

            popt_pl, _ = curve_fit(pl_ccdf, k_fit, F_fit, p0=[1.0, 1.0], maxfev=20000)
            C_pl, alpha_pl = popt_pl
            logF_pl = np.log10(pl_ccdf(k_fit, *popt_pl))
            r2_pl = r2_log(logF, logF_pl)
            print(f"    Power-law: F(k) = {C_pl:.4e} * k^(-{alpha_pl:.4f}), "
                  f"R^2(log–log) = {r2_pl:.4f}")
            ccdf_fits['Power-law'] = (pl_ccdf, popt_pl)

            # Exponential CCDF
            def exp_ccdf(k, C, λ):
                return C*np.exp(-λ*k)

            popt_exp, _ = curve_fit(exp_ccdf, k_fit, F_fit, p0=[1.0, 0.1], maxfev=20000)
            C_exp, λ_exp = popt_exp
            logF_exp = np.log10(exp_ccdf(k_fit, *popt_exp))
            r2_exp = r2_log(logF, logF_exp)
            print(f"    Exponential: F(k) = {C_exp:.4e} * exp(-{λ_exp:.4f} * k), "
                  f"R^2(log–log) = {r2_exp:.4f}")
            ccdf_fits['Exponential'] = (exp_ccdf, popt_exp)

            # Log-normal CCDF
            from scipy.stats import norm

            def lognormal_ccdf(k, A, mu, sigma):
                z = (np.log(k) - mu) / sigma
                return A * (1 - norm.cdf(z))

            mu0 = np.mean(np.log(k_fit))
            sigma0 = np.std(np.log(k_fit))
            A0 = 1.0
            popt_logn, _ = curve_fit(lognormal_ccdf, k_fit, F_fit,
                                     p0=[A0, mu0, sigma0], maxfev=30000)
            A_logn, mu_logn, sigma_logn = popt_logn
            logF_logn = np.log10(lognormal_ccdf(k_fit, *popt_logn))
            r2_logn = r2_log(logF, logF_logn)
            print(f"    Log-normal: A={A_logn:.4e}, mu={mu_logn:.4f}, "
                  f"sigma={sigma_logn:.4f}, R^2(log–log) = {r2_logn:.4f}")
            ccdf_fits['Log-normal'] = (lognormal_ccdf, popt_logn)

    except Exception as e:
        print(f"  CCDF fitting failed: {e}")

    # Combined PDF and CCDF
    k_plot = np.linspace(min(degrees_unique), max(degrees_unique), 400)

    fig, (ax_pdf, ax_ccdf) = plt.subplots(2, 1, figsize=(10, 10))

    # PDF  
    ax_pdf.bar(degrees_unique, prob, width=0.8, alpha=0.7, color='steelblue', 
               edgecolor='black', label='Empirical PDF (Data)')
    for name, (func, params) in pdf_fits.items():
        ax_pdf.plot(k_plot, func(k_plot, *params), lw=2, label=f'{name} fit')
    ax_pdf.set_ylabel('Probability P(k)', fontsize=24, family='serif')
    ax_pdf.set_title('Degree Distribution (PDF & CCDF)', fontsize=30, family='serif')
    ax_pdf.grid(True, alpha=0.3)
    ax_pdf.tick_params(axis='both', which='major', labelsize=20)
    ax_pdf.legend(prop={'family': 'serif', 'size': 20})

    # CCDF 
    ax_ccdf.loglog(degrees_unique, ccdf, 'ko', alpha=0.7, markersize=5,
                   label='Empirical CCDF (Data)')
    for name, (func, params) in ccdf_fits.items():
        ax_ccdf.loglog(k_plot, func(k_plot, *params), lw=2, label=f'{name} fit')
    ax_ccdf.set_xlabel('Degree (k)', fontsize=24, family='serif')
    ax_ccdf.set_ylabel('P(K ≥ k)', fontsize=24, family='serif')
    ax_ccdf.grid(True, alpha=0.3, which='both')
    ax_ccdf.tick_params(axis='both', which='major', labelsize=20)
    ax_ccdf.legend(prop={'family': 'serif', 'size': 20})

    plt.tight_layout()
    plt.savefig('degree_pdf_ccdf_fits.pdf', dpi=300, bbox_inches='tight')
    plt.close()


# SECTION 6: CENTRALITY MEASURES
# Compute multiple centrality measures and rank top 25 nodes by each.
def analyse_centrality(G, largest_cc):
    print("SECTION 6: Centrality Structure")
    print("-"*70)

    centrality_results = {}

    # 6.1 DEGREE CENTRALITY-
    print("6.1 Degree Centrality")
    degree_centrality = nx.degree_centrality(G)
    top_degree = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:25]
    centrality_results['degree'] = degree_centrality

    # 6.2 CLOSENESS CENTRALITY
    print("6.2 Closeness Centrality")
    closeness_centrality = nx.closeness_centrality(largest_cc)
    top_closeness = sorted(closeness_centrality.items(), key=lambda x: x[1], reverse=True)[:25]
    centrality_results['closeness'] = closeness_centrality

    # 6.3 BETWEENNESS CENTRALITY
    print("6.3 Betweenness Centrality")
    betweenness_centrality = nx.betweenness_centrality(G, normalized=True)
    top_betweenness = sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:25]
    centrality_results['betweenness'] = betweenness_centrality

    # 6.4 EIGENVECTOR CENTRALITY
    print("6.4 Eigenvector Centrality")
    eigenvector_centrality = nx.eigenvector_centrality(largest_cc, max_iter=1000)
    top_eigenvector = sorted(eigenvector_centrality.items(), key=lambda x: x[1], reverse=True)[:25]
    centrality_results['eigenvector'] = eigenvector_centrality

    # 6.5 KATZ CENTRALITY
    print("6.5 Katz Centrality")
    katz_centrality = nx.katz_centrality(largest_cc, alpha=0.005, beta=1.0, max_iter=1000, tol=1e-06)
    top_katz = sorted(katz_centrality.items(), key=lambda x: x[1], reverse=True)[:25]
    centrality_results['katz'] = katz_centrality

    # 6.6 PAGERANK
    print("6.6 PageRank")
    pagerank = nx.pagerank(G, alpha=0.85, max_iter=100)
    top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:25]
    centrality_results['pagerank'] = pagerank

    # 6.7 SUBGRAPH CENTRALITY
    print("6.7 Subgraph Centrality")
    subgraph_centrality = nx.subgraph_centrality(largest_cc)
    top_subgraph = sorted(subgraph_centrality.items(), key=lambda x: x[1], reverse=True)[:25]

    centrality_results['subgraph'] = subgraph_centrality

    # 6.8 SAVE TOP 25 RANKINGS
    print("6.8 Top 25 Rankings")
    top_25_dict = {
        'Degree': [node for node, _ in top_degree],
        'Closeness': [node for node, _ in top_closeness],
        'Betweenness': [node for node, _ in top_betweenness],
        'Eigenvector': [node for node, _ in top_eigenvector],
        'Katz': [node for node, _ in top_katz],
        'PageRank': [node for node, _ in top_pagerank],
        'Subgraph': [node for node, _ in top_subgraph],
    }

    top_25_df = pd.DataFrame(top_25_dict)
    top_25_df.index = range(1, 26)
    top_25_df.index.name = 'Rank'
    top_25_df.to_csv('centrality_top25_rankings.csv', index=True)

    # 6.9 RANKING COMPARISON ANALYSIS
    print("6.9 Analyzing Ranking Overlaps using Jaccard Similarity")
    available_measures = [
        m for m in
        ['degree', 'closeness', 'betweenness', 'eigenvector',
         'katz', 'pagerank', 'subgraph']
        if centrality_results[m] is not None
    ]

    overlap_matrix = np.zeros((len(available_measures), len(available_measures)))

    for i, measure1 in enumerate(available_measures):
        measure1_key = measure1.capitalize() if measure1 != 'pagerank' else 'PageRank'
        set1 = set(top_25_dict[measure1_key])
        for j, measure2 in enumerate(available_measures):
            measure2_key = measure2.capitalize() if measure2 != 'pagerank' else 'PageRank'
            set2 = set(top_25_dict[measure2_key])
            inter = len(set1 & set2)
            union = len(set1 | set2)
            jaccard = inter / union if union > 0 else 0.0
            overlap_matrix[i, j] = jaccard

    readable_labels = [
        m.capitalize() if m != 'pagerank' else 'PageRank'
        for m in available_measures
    ]

    overlap_df = pd.DataFrame(
        overlap_matrix, index=readable_labels, columns=readable_labels
    )
    overlap_df.to_csv('centrality_overlap_matrix.csv')

    # 6.10 OVERLAP MATRIX
    print("6.10 Overlap Matrix")
    # Convert Jaccard similarities to overlap counts out of 25
    n_top = 25
    overlap_counts = np.zeros_like(overlap_matrix)
    for i in range(len(available_measures)):
        for j in range(len(available_measures)):
            jaccard = overlap_matrix[i, j]
            # For top-n lists of size n: k=J*(2n)/(1+J)
            k = (2*n_top*jaccard)/(1+jaccard) if jaccard>0 else 0
            overlap_counts[i, j] = int(round(k))

    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(overlap_counts, cmap='hot', vmin=0, vmax=n_top)

    ax.set_xticks(np.arange(len(readable_labels)))
    ax.set_yticks(np.arange(len(readable_labels)))
    ax.set_xticklabels(readable_labels, rotation=45, ha='right',
                       fontsize=16, family='serif')
    ax.set_yticklabels(readable_labels, fontsize=16, family='serif')

    ax.set_xticks(np.arange(len(readable_labels)) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(readable_labels)) - 0.5, minor=True)
    ax.grid(which='minor', color='white', linestyle='-', linewidth=2)

    for i in range(len(readable_labels)):
        for j in range(len(readable_labels)):
            val = overlap_counts[i, j]
            text_color = 'white' if overlap_counts[i,j] < 2 else 'black'
            ax.text(j, i, f"{int(val)}", ha='center', va='center', color=text_color, fontsize=14, fontweight='bold')

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Overlap count (out of 25)', rotation=270, labelpad=20, fontsize=20, family="serif")
    cbar.ax.tick_params(labelsize=20)

    ax.set_title('Overlap Matrix of Top 25 Nodes in Centrality Measures', fontsize=24, pad=20, family='serif')

    plt.tight_layout()
    plt.savefig('centrality_overlap_matrix_counts_heatmap.pdf', dpi=300, bbox_inches='tight')
    plt.close()

    # 6.11 SAVE DATA
    print("6.11 Saving Centrality Data")
    print()

    for measure in available_measures:
        cent_dict = centrality_results[measure]
        df = pd.DataFrame({
            'Node': list(cent_dict.keys()),
            f'{measure.capitalize()}_Centrality': list(cent_dict.values())
        })
        df = df.sort_values(f'{measure.capitalize()}_Centrality', ascending=False)
        df.to_csv(f'{measure}_centrality.csv', index=False)


# SECTION 7: NULL MODELS
# Compare network to null models.
def analyse_null_models(G_real):
    print("SECTION 7: Null Model Analysis")
    print("-"*70)

    # 7.1 REAL NETWORK
    print("7.1 Western US Power Grid")

    # Basic real-network descriptors
    n = G_real.number_of_nodes()
    m = G_real.number_of_edges()
    avg_degree_real = 2*m/n
    density_real = nx.density(G_real)
    avg_clust_real = nx.average_clustering(G_real)

    # Giant component for distances
    if nx.is_connected(G_real):
        GCC_real = G_real
    else:
        gcc_nodes = max(nx.connected_components(G_real), key=len)
        GCC_real = G_real.subgraph(gcc_nodes).copy()

    avg_path_real = nx.average_shortest_path_length(GCC_real)
    diam_real = nx.diameter(GCC_real)

    # Louvain modularity for real network
    part_real = community_louvain.best_partition(G_real)
    Q_real = community_louvain.modularity(part_real, G_real)

    print(f"Real network: n={n}, m={m}")
    print(f"  Avg degree      : {avg_degree_real:.4f}")
    print(f"  Density         : {density_real:.6f}")
    print(f"  Clustering      : {avg_clust_real:.6f}")
    print(f"  Avg path length : {avg_path_real:.6f}")
    print(f"  Diameter (GCC)  : {diam_real:.4f}")
    print(f"  Modularity Q    : {Q_real:.4f}")
    print()

    # Calculate basic metrics
    def compute_metrics(H):
        
        nH = H.number_of_nodes()
        mH = H.number_of_edges()
        avg_deg = 2*mH/nH
        ρ = nx.density(H)
        C = nx.average_clustering(H)

        if nx.is_connected(H):
            GCC = H
        else:
            gcc_nodes = max(nx.connected_components(H), key=len)
            GCC = H.subgraph(gcc_nodes).copy()

        L = nx.average_shortest_path_length(GCC)
        D = nx.diameter(GCC)

        part = community_louvain.best_partition(H)
        Q = community_louvain.modularity(part, H)

        return avg_deg, ρ, C, L, D, Q

    # Determine top-25 nodes of several centralities
    def centrality_top25_overlap(H):
        print("Calculating centrality measures")
        measures = {
            "Degree": nx.degree_centrality(H),
            "Closeness": nx.closeness_centrality(H),
            "Betweenness": nx.betweenness_centrality(H),
            "Eigenvector": nx.eigenvector_centrality(H, max_iter=1000),
            "Katz": nx.katz_centrality(H, alpha=0.005, beta=1.0, max_iter=1000),
            "PageRank": nx.pagerank(H),
            "Subgraph": nx.subgraph_centrality(H),
        }
        names = list(measures.keys())
        top_sets = {}
        for name, vals in measures.items():
            top_nodes = sorted(vals.items(), key=lambda x: x[1], reverse=True)[:25]
            top_sets[name] = {u for u, _ in top_nodes}

        k = len(names)
        overlap = np.zeros((k, k), dtype=float)
        for i, ni in enumerate(names):
            for j, nj in enumerate(names):
                overlap[i, j] = len(top_sets[ni] & top_sets[nj])
        return names, overlap

    # Plot overlap matrix
    def plot_overlap_matrix(names, overlap, title, filename):
        print("Plotting overlap matrix")
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(overlap, cmap='hot')

        # Set major ticks at cell centers
        ax.set_xticks(np.arange(len(names)))
        ax.set_yticks(np.arange(len(names)))

        ax.set_xticks(np.arange(len(names))-0.5, minor=True)
        ax.set_xticklabels(names, rotation=45, ha='right', fontsize=16, family='serif')
        ax.set_yticks(np.arange(len(names))-0.5, minor=True)
        ax.set_yticklabels(names, fontsize=16, family='serif')
        ax.grid(which='minor', color='white', linestyle='-', linewidth=2)

        for i in range(len(names)):
            for j in range(len(names)):
                text_color = 'white' if overlap[i,j] < 10 else 'black'
                ax.text(j, i, f"{int(overlap[i, j])}",
                        ha='center', va='center', color=text_color,
                        fontsize=10, family='serif', fontweight='bold')

        ax.set_title(title, fontsize=20, family='serif')
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Overlap count (out of 25)', rotation=270, labelpad=20, fontsize=20, family="serif")
        cbar.ax.tick_params(labelsize=16)
        plt.tight_layout()
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()

    # Compute R^2
    def r2(y_true, y_pred):
        ss_res = np.sum((y_true-y_pred)**2)
        ss_tot = np.sum((y_true-np.mean(y_true))**2)
        return 1-ss_res/ss_tot if ss_tot>0 else np.nan

    # Compute R^2 in log space
    def r2_log(y_true_log, y_pred_log):
        ss_res = np.sum((y_true_log-y_pred_log)**2)
        ss_tot = np.sum((y_true_log-np.mean(y_true_log))**2)
        return 1-ss_res/ss_tot if ss_tot>0 else np.nan

    # Mean/std PDF & CCDF
    def degree_histogram_stats(G_list, bins):
        print("Computing degree histogram statistics")
        pdf_all = []
        ccdf_all = []

        for G in G_list:
            degrees = np.array([d for _, d in G.degree()])
            hist, edges = np.histogram(degrees, bins=bins, density=True)
            pdf_all.append(hist)

            cdf = np.cumsum(hist)*(edges[1]-edges[0])
            ccdf = 1.0-cdf
            ccdf_all.append(ccdf)

        pdf_all = np.vstack(pdf_all)
        ccdf_all = np.vstack(ccdf_all)

        pdf_mean = pdf_all.mean(axis=0)
        pdf_std = pdf_all.std(axis=0, ddof=1)
        ccdf_mean = ccdf_all.mean(axis=0)
        ccdf_std = ccdf_all.std(axis=0, ddof=1)
        bin_centers = 0.5*(edges[:-1]+edges[1:])
        return bin_centers, pdf_mean, pdf_std, ccdf_mean, ccdf_std

    # Plot PDF and CCDF with fits
    def plot_distributions(bin_centers, pdf_mean, pdf_std, ccdf_mean, ccdf_std, title, filename):  
        print("Plotting degree distributions with fits")      
        fig, (ax_pdf, ax_ccdf) = plt.subplots(2, 1, figsize=(10, 10))

        # PDF 
        ax_pdf.bar(bin_centers, pdf_mean, width=0.8, alpha=0.7, color='steelblue', edgecolor='black', label='Mean PDF ± std')
        ax_pdf.errorbar(bin_centers, pdf_mean, yerr=pdf_std, fmt='none', ecolor='black', capsize=3, alpha=0.8)

        threshold = 1e-3  
        mask = pdf_mean > threshold
        if mask.any():
            x_max_plot_pdf = bin_centers[mask].max()
        else:
            x_max_plot_pdf = bin_centers.max()

        # Fit PDF
        pdf_fit_x = bin_centers[pdf_mean > 0]
        pdf_fit_y = pdf_mean[pdf_mean > 0]

        if len(pdf_fit_x)>5:
            # Power-law PDF
            def power_law_pdf(k, C, gamma):
                return C*k**(-gamma)

            try:
                popt_pl, _ = curve_fit(power_law_pdf, pdf_fit_x, pdf_fit_y, p0=[1.0, 2.0], maxfev=20000)
                p_pred_pl = power_law_pdf(pdf_fit_x, *popt_pl)
                r2_pl = r2(pdf_fit_y, p_pred_pl)
                k_plot = np.linspace(pdf_fit_x.min(), pdf_fit_x.max(), 200)
                ax_pdf.plot(k_plot, power_law_pdf(k_plot, *popt_pl), 'b-', lw=2, label=f'Power-law fit (R²={r2_pl:.3f})')
            except:
                pass

            # Exponential PDF
            def exp_pdf(k, C, λ):
                return C*np.exp(-λ*k)

            try:
                popt_exp, _ = curve_fit(exp_pdf, pdf_fit_x, pdf_fit_y, p0=[1.0, 0.1], maxfev=20000)
                p_pred_exp = exp_pdf(pdf_fit_x, *popt_exp)
                r2_exp = r2(pdf_fit_y, p_pred_exp)
                k_plot = np.linspace(pdf_fit_x.min(), pdf_fit_x.max(), 200)
                ax_pdf.plot(k_plot, exp_pdf(k_plot, *popt_exp), 'orange', lw=2, label=f'Exponential fit (R²={r2_exp:.3f})')
            except:
                pass

            # Log-normal PDF
            def lognormal_pdf(k, A, mu, sigma):
                return A*(1.0/(k*sigma*np.sqrt(2*np.pi)))*\
                       np.exp(-(np.log(k)-mu)**2/(2*sigma**2))

            try:
                logk = np.log(pdf_fit_x)
                p0_logn = [max(pdf_fit_y), np.mean(logk), 2.0*np.std(logk)]
                popt_logn, _ = curve_fit(
                    lognormal_pdf, 
                    pdf_fit_x, 
                    pdf_fit_y,
                    p0=p0_logn, 
                    bounds=([0, -np.inf, 1e-6], [np.inf, np.inf, np.inf]),
                    maxfev=100000  
                )
                p_pred_logn = lognormal_pdf(pdf_fit_x, *popt_logn)
                r2_logn = r2(pdf_fit_y, p_pred_logn)
                k_plot = np.linspace(pdf_fit_x.min(), pdf_fit_x.max(), 200)
                ax_pdf.plot(k_plot, lognormal_pdf(k_plot, *popt_logn), 'g-', lw=2, label=f'Log-normal fit (R²={r2_logn:.3f})')
            except Exception as e:
                print(f"Log-normal fit failed: {e}") 
                pass

        ax_pdf.set_xlim(left=0, right=x_max_plot_pdf*1.1)
        ax_pdf.set_ylabel('Probability P(k)', fontsize=20, family='serif')
        ax_pdf.set_title(f'{title} Degree Distribution (PDF & CCDF)', fontsize=24, family='serif')
        ax_pdf.grid(True, alpha=0.3)
        ax_pdf.tick_params(axis='both', which='major', labelsize=20)
        ax_pdf.legend(prop={'family': 'serif', 'size': 20})

        # CCDF - Log-log scale with fits
        ax_ccdf.errorbar(bin_centers, ccdf_mean, yerr=ccdf_std, fmt='o', color='black', ecolor='black', capsize=3, label='Mean CCDF ± std', alpha=0.8, markersize=5)

        # Fit CCDF
        ccdf_fit_x = bin_centers[(ccdf_mean>0) & (bin_centers>0)]
        ccdf_fit_y = ccdf_mean[(ccdf_mean>0) & (bin_centers>0)]

        if len(ccdf_fit_x)>5:
            logF = np.log10(ccdf_fit_y)

            threshold = 1e-3
            mask = ccdf_fit_y>threshold
            if mask.any():
                x_max_plot = ccdf_fit_x[mask].max()
            else:
                x_max_plot = ccdf_fit_x.max()

            # Power-law CCDF
            def pl_ccdf(k, C, alpha):
                return C*k**(-alpha)

            try:
                popt_pl, _ = curve_fit(pl_ccdf, ccdf_fit_x, ccdf_fit_y, p0=[1.0, 1.0], maxfev=20000)
                logF_pl = np.log10(pl_ccdf(ccdf_fit_x, *popt_pl))
                r2_pl_ccdf = r2_log(logF, logF_pl)
                k_plot = np.logspace(np.log10(ccdf_fit_x.min()), np.log10(ccdf_fit_x.max()), 200)
                ax_ccdf.loglog(k_plot, pl_ccdf(k_plot, *popt_pl), 'b-', lw=2, label=f'Power-law fit (R²={r2_pl_ccdf:.3f})')
            except:
                pass

            # Exponential CCDF
            def exp_ccdf(k, C, λ):
                return C*np.exp(-λ*k)

            try:
                popt_exp, _ = curve_fit(exp_ccdf, ccdf_fit_x, ccdf_fit_y, p0=[1.0, 0.1], maxfev=20000)
                logF_exp = np.log10(exp_ccdf(ccdf_fit_x, *popt_exp))
                r2_exp_ccdf = r2_log(logF, logF_exp)
                k_plot = np.logspace(np.log10(ccdf_fit_x.min()), np.log10(ccdf_fit_x.max()), 200)
                ax_ccdf.loglog(k_plot, exp_ccdf(k_plot, *popt_exp), 'orange', lw=2, label=f'Exponential fit (R²={r2_exp_ccdf:.3f})')
            except:
                pass

            # Log-normal CCDF
            from scipy.stats import norm
            
            def lognormal_ccdf(k, A, mu, sigma):
                z = (np.log(k)-mu)/sigma
                return A*(1-norm.cdf(z))

            try:
                mu0 = np.mean(np.log(ccdf_fit_x))
                sigma0 = np.std(np.log(ccdf_fit_x))
                A0 = 1.0
                popt_logn, _ = curve_fit(lognormal_ccdf, ccdf_fit_x, ccdf_fit_y, p0=[A0, mu0, sigma0], maxfev=30000)
                logF_logn = np.log10(lognormal_ccdf(ccdf_fit_x, *popt_logn))
                r2_logn_ccdf = r2_log(logF, logF_logn)
                k_plot = np.logspace(np.log10(ccdf_fit_x.min()), np.log10(ccdf_fit_x.max()), 200)
                ax_ccdf.loglog(k_plot, lognormal_ccdf(k_plot, *popt_logn), 'g-', lw=2, label=f'Log-normal fit (R²={r2_logn_ccdf:.3f})')
            except:
                pass
        
        if len(ccdf_fit_x)>5:
            ax_ccdf.set_xlim(left=ccdf_fit_x.min(), right=x_max_plot*1.1)

        ax_ccdf.set_xlabel('Degree (k)', fontsize=20, family='serif')
        ax_ccdf.set_ylabel('P(K ≥ k)', fontsize=20, family='serif')
        ax_ccdf.grid(True, which='both', alpha=0.3)
        ax_ccdf.tick_params(axis='both', which='major', labelsize=20)
        ax_ccdf.legend(prop={'family': 'serif', 'size': 20})

        plt.tight_layout()
        plt.savefig(f'{filename}_degree_fits.pdf', dpi=300, bbox_inches='tight')
        plt.close()

    # Real Overlap Matrix
    names_real, overlap_real = centrality_top25_overlap(G_real)
    plot_overlap_matrix(names_real, overlap_real, "Real network overlap (top-25 centralities)", "real_overlap_matrix.pdf")


    # 7.2 ER NETWORKS
    print("7.2 Erdős–Rényi Graphs")
    p = 2*m/(n*(n-1))

    er_metrics = []
    er_graphs = []
    er_overlaps = []

    for i in range(10):
        G_er = nx.gnp_random_graph(n, p, seed=42+i)
        er_graphs.append(G_er)

        metrics = compute_metrics(G_er)
        er_metrics.append(metrics)

        _, overlap = centrality_top25_overlap(G_er)
        er_overlaps.append(overlap)

    er_metrics = np.array(er_metrics)
    er_mean = er_metrics.mean(axis=0)
    er_std  = er_metrics.std(axis=0, ddof=1)

    # ER Metrics 
    print(f"  Avg degree      : {er_mean[0]:.4f} ± {er_std[0]:.4f}")
    print(f"  Density         : {er_mean[1]:.6f} ± {er_std[1]:.6f}")
    print(f"  Clustering      : {er_mean[2]:.6f} ± {er_std[2]:.6f}")
    print(f"  Avg path length : {er_mean[3]:.6f} ± {er_std[3]:.6f}")
    print(f"  Diameter (GCC)  : {er_mean[4]:.4f} ± {er_std[4]:.4f}")
    print(f"  Modularity Q    : {er_mean[5]:.4f} ± {er_std[5]:.4f}")
    print()

    er_overlap_mean = np.mean(er_overlaps, axis=0)
    plot_overlap_matrix(names_real, er_overlap_mean, "ER mean overlap matrix (top-25 centralities)", "er_overlap_mean.pdf")

    # ER degree distributions 
    max_deg_real = max(d for _, d in G_real.degree())
    bins = np.arange(1, max_deg_real+2)
    bc_er, pdf_m_er, pdf_s_er, ccdf_m_er, ccdf_s_er = degree_histogram_stats(er_graphs, bins)
    plot_distributions(bc_er, pdf_m_er, pdf_s_er, ccdf_m_er, ccdf_s_er, "ER Networks", "er_degree")


    # 7.3 BA NETWORKS
    print("7.3 Barabási–Albert Graphs")
    m0 = max(1, int(round(m/n)))
    m0 = min(m0, n-1)

    # ba_metrics = []
    ba_graphs = []
    ba_overlaps = []

    for i in range(10):
        G_ba = nx.barabasi_albert_graph(n, m0, seed=100+i)
        ba_graphs.append(G_ba)

        metrics = compute_metrics(G_ba)
        ba_metrics.append(metrics)

        _, overlap = centrality_top25_overlap(G_ba)
        ba_overlaps.append(overlap)

    ba_metrics = np.array(ba_metrics)
    ba_mean = ba_metrics.mean(axis=0)
    ba_std  = ba_metrics.std(axis=0, ddof=1)

    # BA Metrics 
    print(f"  Avg degree      : {ba_mean[0]:.4f} ± {ba_std[0]:.4f}")
    print(f"  Density         : {ba_mean[1]:.6f} ± {ba_std[1]:.6f}")
    print(f"  Clustering      : {ba_mean[2]:.6f} ± {ba_std[2]:.6f}")
    print(f"  Avg path length : {ba_mean[3]:.6f} ± {ba_std[3]:.6f}")
    print(f"  Diameter (GCC)  : {ba_mean[4]:.4f} ± {ba_std[4]:.4f}")
    print(f"  Modularity Q    : {ba_mean[5]:.4f} ± {ba_std[5]:.4f}")
    print()

    ba_overlap_mean = np.mean(ba_overlaps, axis=0)
    plot_overlap_matrix(names_real, ba_overlap_mean, "BA mean overlap matrix (top-25 centralities)", "ba_overlap_mean.pdf")

    # BA degree distributions 
    bc_ba, pdf_m_ba, pdf_s_ba, ccdf_m_ba, ccdf_s_ba = degree_histogram_stats(ba_graphs, bins)
    plot_distributions(bc_ba, pdf_m_ba, pdf_s_ba, ccdf_m_ba, ccdf_s_ba, "BA Networks", "ba_degree")


    # 7.4 COMPARISON TABLE
    print("7.4 Comparison Table")
    comparison_df = pd.DataFrame({
        "Network": ["Real", "ER (mean)", "BA (mean)"],
        "Avg_Degree":      [avg_degree_real, er_mean[0],  ba_mean[0]],
        "Avg_Degree_std":  [0.0,             er_std[0],   ba_std[0]],
        "Density":         [density_real,    er_mean[1],  ba_mean[1]],
        "Density_std":     [0.0,             er_std[1],   ba_std[1]],
        "Clustering":      [avg_clust_real,  er_mean[2],  ba_mean[2]],
        "Clustering_std":  [0.0,             er_std[2],   ba_std[2]],
        "Avg_Path_Length": [avg_path_real,   er_mean[3],  ba_mean[3]],
        "Avg_Path_Length_std": [0.0,         er_std[3],   ba_std[3]],
        "Diameter":        [diam_real,       er_mean[4],  ba_mean[4]],
        "Diameter_std":    [0.0,             er_std[4],   ba_std[4]],
        "Modularity_Q":    [Q_real,          er_mean[5],  ba_mean[5]],
        "Modularity_Q_std":[0.0,             er_std[5],   ba_std[5]],
    })

    comparison_df.to_csv("null_model_comparison_ER_BA.csv", index=False)
    print("Comparison Table")
    print(comparison_df.to_string(index=False))
    return comparison_df


# SECTION 8: COMMUNITY DETECTION
# Detect communities using multiple algorithms.
def analyse_communities(G):
    print("SECTION 8: Community Detection")
    print("-"*70)
    print()
    
    results = {}
    all_methods = {}

    # Quality metrics for commutity detection methods 
    def compute_quality_metrics(G, partition):
        metrics = {}
        
        # Convert partition dict to list of sets for NetworkX functions
        comm_dict = {}
        for node, comm_id in partition.items():
            if comm_id not in comm_dict:
                comm_dict[comm_id] = set()
            comm_dict[comm_id].add(node)
        communities_list = list(comm_dict.values())
        
        # 1. Modularity 
        metrics['modularity'] = nx.community.modularity(G, communities_list)

        # 2. Number of communities
        metrics['n_communities'] = len(communities_list)
        
        # 3. Size statistics
        sizes = [len(c) for c in communities_list]
        metrics['largest_community'] = max(sizes)
        metrics['smallest_community'] = min(sizes)
        metrics['avg_community_size'] = np.mean(sizes)
        metrics['std_community_size'] = np.std(sizes)
        
        # 4. Edge density within communities 
        internal_edges = 0
        total_possible_edges = 0
        for comm in communities_list:
            comm_subgraph = G.subgraph(comm)
            internal_edges += comm_subgraph.number_of_edges()
            n = len(comm)
            total_possible_edges+=n*(n-1)/2
        
        metrics['internal_edge_density'] = (
            internal_edges/total_possible_edges if total_possible_edges>0 else 0
        )
        
        # 5. Conductance 
        cut_edges = 0
        for comm in communities_list:
            for node in comm:
                for neighbor in G.neighbors(node):
                    if neighbor not in comm:
                        cut_edges += 1
        
        internal_degree_sum = 2 * internal_edges
        metrics['conductance'] = (
            cut_edges / (internal_degree_sum + cut_edges) 
            if (internal_degree_sum + cut_edges) > 0 else 0
        )
        
        return metrics, communities_list
    
    # 8.1 LOUVAIN ALGORITHM 
    print("8.1 Louvain Algorithm")
    import community.community_louvain as community_louvain
    
    partition_louvain = community_louvain.best_partition(G, random_state=42)
    metrics_louvain, comms_louvain = compute_quality_metrics(G, partition_louvain)
    
    all_methods['LA'] = {
        'partition': partition_louvain,
        'metrics': metrics_louvain,
        'communities': comms_louvain
    }
    
    print(f"   Communities: {metrics_louvain['n_communities']}")
    print(f"   Modularity Q: {metrics_louvain['modularity']:.4f}")
    print(f"   Internal Edge Density: {metrics_louvain['internal_edge_density']:.4f}")
    print(f"   Conductance: {metrics_louvain['conductance']:.4f}")
    print()
    
    # 8.2 GREEDY MODULARITY OPTIMIZATION
    print("8.2 Greedy Modularity Optimization")
    communities_greedy = list(
        nx.community.greedy_modularity_communities(G, weight=None)
    )
    
    # Convert to dict
    partition_greedy = {}
    for cid, comm in enumerate(communities_greedy):
        for node in comm:
            partition_greedy[node] = cid
    
    metrics_greedy, comms_greedy = compute_quality_metrics(G, partition_greedy)
    
    all_methods['GM'] = {
        'partition': partition_greedy,
        'metrics': metrics_greedy,
        'communities': comms_greedy
    }
    
    print(f"   Communities: {metrics_greedy['n_communities']}")
    print(f"   Modularity Q: {metrics_greedy['modularity']:.4f}")
    print(f"   Internal Edge Density: {metrics_greedy['internal_edge_density']:.4f}")
    print(f"   Conductance: {metrics_greedy['conductance']:.4f}")
    print()
        
    # 8.3 LABEL PROPAGATION
    print("8.3 Label Propagation")
    communities_lp = list(nx.community.label_propagation_communities(G))
    
    partition_lp = {}
    for cid, comm in enumerate(communities_lp):
        for node in comm:
            partition_lp[node] = cid
    
    metrics_lp, comms_lp = compute_quality_metrics(G, partition_lp)
    
    all_methods['LP'] = {
        'partition': partition_lp,
        'metrics': metrics_lp,
        'communities': comms_lp
    }
    
    print(f"   Communities: {metrics_lp['n_communities']}")
    print(f"   Modularity Q: {metrics_lp['modularity']:.4f}")
    print(f"   Internal Edge Density: {metrics_lp['internal_edge_density']:.4f}")
    print(f"   Conductance: {metrics_lp['conductance']:.4f}")
    print()
    
    # 8.4 ASYNCHRONOUS LABEL PROPAGATION
    print("8.4 Asynchronous Label Propagation")
    communities_alp = list(nx.community.asyn_lpa_communities(G, seed=42))
    
    partition_alp = {}
    for cid, comm in enumerate(communities_alp):
        for node in comm:
            partition_alp[node] = cid
    
    metrics_alp, comms_alp = compute_quality_metrics(G, partition_alp)
    
    all_methods['ALP'] = {
        'partition': partition_alp,
        'metrics': metrics_alp,
        'communities': comms_alp
    }
    
    print(f"   Communities: {metrics_alp['n_communities']}")
    print(f"   Modularity Q: {metrics_alp['modularity']:.4f}")
    print(f"   Internal Edge Density: {metrics_alp['internal_edge_density']:.4f}")
    print(f"   Conductance: {metrics_alp['conductance']:.4f}")
    print()
    
    # 8.5 FLUID COMMUNITIES
    print("8.5 Fluid Communities")
    # Fluid communities requires specifying k
    k_fluid = max(3, int(np.sqrt(G.number_of_nodes() / 10)))
    communities_fluid = list(nx.community.asyn_fluidc(G, k_fluid, seed=42))
    
    partition_fluid = {}
    for cid, comm in enumerate(communities_fluid):
        for node in comm:
            partition_fluid[node] = cid
    
    metrics_fluid, comms_fluid = compute_quality_metrics(G, partition_fluid)
    
    all_methods['FC'] = {
        'partition': partition_fluid,
        'metrics': metrics_fluid,
        'communities': comms_fluid
    }
    
    print(f"   Communities: {metrics_fluid['n_communities']}")
    print(f"   Modularity Q: {metrics_fluid['modularity']:.4f}")
    print(f"   Internal Edge Density: {metrics_fluid['internal_edge_density']:.4f}")
    print(f"   Conductance: {metrics_fluid['conductance']:.4f}")
    print()
    
    # 8.6 COMPARISON OF METHODS
    print("8.6 Comparison of methods")
    print()
    
    # Create comparison table
    comparison_data = []
    for method_name, method_data in all_methods.items():
        comparison_data.append({
            'Method': method_name,
            'N_Communities': method_data['metrics']['n_communities'],
            'Modularity_Q': method_data['metrics']['modularity'],
            'Internal_Density': method_data['metrics']['internal_edge_density'],
            'Conductance': method_data['metrics']['conductance'],
            'Avg_Size': int(method_data['metrics']['avg_community_size']),
            'Std_Size': int(method_data['metrics']['std_community_size'])
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('Modularity_Q', ascending=False)
    
    print(comparison_df.to_string(index=False))
    print()
    comparison_df.to_csv('community_methods_comparison.csv', index=False)
    
    # Select best method by modularity
    best_method_name = comparison_df.iloc[0]['Method']
    best_method = all_methods[best_method_name]
    
    # 8.7 BEST METHOD
    print("8.7 Select best method")
    print()
    print(f"Best method: {best_method_name}")
    print()
    print(f"  Modularity Q: {best_method['metrics']['modularity']:.4f}")
    print(f"  Communities: {best_method['metrics']['n_communities']}")
    print(f"  Internal Edge Density: {best_method['metrics']['internal_edge_density']:.4f}")
    print(f"  Conductance: {best_method['metrics']['conductance']:.4f}")
    print(f"  Average Community Size: {best_method['metrics']['avg_community_size']:.1f}")
    print(f"  Std Dev Community Size: {best_method['metrics']['std_community_size']:.1f}")
    print()
    
    # Use best partition 
    partition = best_method['partition']
    
    sizes = Counter(partition.values())
    total_found = len(sizes)
    print(f"Total number of communities detected: {total_found}")
    print("Sizes of all communities (top 15):")
    for cid, sz in sizes.most_common(15):
        print(f"  Community {cid}: {sz} nodes")
    print()
    
    # Save assignments 
    community_df = pd.DataFrame({
        "Node": list(partition.keys()),
        "Community": list(partition.values())
    })
    community_df = community_df.sort_values('Node').reset_index(drop=True)
    community_df.to_csv(
        f"community_assignments_{best_method_name.lower()}.csv", 
        index=False
    )
    
    # 8.8 BONUS: Export GML with community attributes
    print("8.8 BONUS: Export GML with community attributes")
    G_comm = G.copy()
    nx.set_node_attributes(G_comm, partition, name="community")
    
    G_comm.graph["community_method"] = best_method_name
    G_comm.graph["community_modularity"] = float(best_method['metrics']['modularity'])
    G_comm.graph["community_internal_density"] = float(
        best_method['metrics']['internal_edge_density']
    )
    G_comm.graph["community_conductance"] = float(best_method['metrics']['conductance'])
    G_comm.graph["n_communities"] = int(total_found)
    
    out_file = f"network_with_communities_{best_method_name.lower()}.gml"
    nx.write_gml(G_comm, out_file)
    
    # 8.9 SAVE DATA
    print("8.9 Store Results")
    print()
    results["all_methods"] = all_methods
    results["best_method"] = best_method_name
    results["partition"] = partition
    results["metrics"] = best_method['metrics']
    results["comparison_df"] = comparison_df
    results["n_communities"] = total_found  
    return results
# Create plots comparing all community detection methods
def community_comparison(results):    
    print("Visualize comparison between community detection methods")
    comparison_df = results['comparison_df']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    methods = comparison_df['Method'].values
    x_pos = np.arange(len(methods))

    # 8.1.1 MODULARITY
    print("8.1.1 Modularity Plot")
    ax1 = axes[0, 0]
    bars1 = ax1.bar(x_pos, comparison_df['Modularity_Q'], 
                    color='steelblue', edgecolor='black', linewidth=1.5)
    ax1.set_xlabel('Method', fontsize=24, family='serif')
    ax1.set_ylabel('Modularity Q', fontsize=24, family='serif')
    ax1.set_title('Modularity Comparison', fontsize=28, family='serif')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(methods, rotation=45, ha='right', fontsize=24, family='serif')
    ax1.tick_params(axis='both', which='major', labelsize=24)
    ax1.grid(axis='y', alpha=0.3)
    ymin, ymax = ax1.get_ylim()
    ax1.set_ylim(ymin, ymax * 1.08)

    for i, bar in enumerate(bars1):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=24, family='serif')
    
    # 8.1.2 INTERNAL EDGE DENSITY
    print("8.1.2 Internal Edge Density Plot")
    ax2 = axes[0, 1]
    bars2 = ax2.bar(x_pos, comparison_df['Internal_Density'], 
                    color='coral', edgecolor='black', linewidth=1.5)
    ax2.set_xlabel('Method', fontsize=24, family='serif')
    ax2.set_ylabel('Internal Edge Density', fontsize=24, family='serif')
    ax2.set_title('Internal Cohesion', fontsize=28, family='serif')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(methods, rotation=45, ha='right', fontsize=24, family='serif')
    ax2.tick_params(axis='both', which='major', labelsize=24)
    ax2.grid(axis='y', alpha=0.3)
    ymin, ymax = ax2.get_ylim()
    ax2.set_ylim(ymin, ymax * 1.15)
    
    for i, bar in enumerate(bars2):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=24, family='serif')
    
    # 8.1.3 CONDUCTANCE
    print("8.1.3 Conductance Plot")
    ax3 = axes[1, 0]
    bars3 = ax3.bar(x_pos, comparison_df['Conductance'], 
                    color='mediumseagreen', edgecolor='black', linewidth=1.5)
    ax3.set_xlabel('Method', family='serif', fontsize=24)
    ax3.set_ylabel('Conductance', family='serif', fontsize=24)
    ax3.set_title('Community Isolation', family='serif', fontsize=28)
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(methods, rotation=45, ha='right', fontsize=24, family='serif')
    ax3.tick_params(axis='both', which='major', labelsize=24)
    ax3.grid(axis='y', alpha=0.3)
    ymin, ymax = ax3.get_ylim()
    ax3.set_ylim(ymin, ymax * 1.15)
    
    for i, bar in enumerate(bars3):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=24, family='serif')
    
    # 8.1.4 NUMBER OF COMMUNITIES
    print("8.1.4 Number of Communities Plot")
    ax4 = axes[1, 1]
    bars4 = ax4.bar(x_pos, comparison_df['N_Communities'], 
                    color='mediumpurple', edgecolor='black', linewidth=1.5)
    ax4.set_xlabel('Method', family='serif', fontsize=24)
    ax4.set_ylabel('Nº of Communities', family='serif', fontsize=24)
    ax4.set_title('Partition Granularity', family='serif', fontsize=28)
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(methods, rotation=45, ha='right', fontsize=24, family='serif')
    ax4.tick_params(axis='both', which='major', labelsize=24)
    ax4.grid(axis='y', alpha=0.3)
    ymin, ymax = ax4.get_ylim()
    ax4.set_ylim(ymin, ymax*1.08)
    
    for i, bar in enumerate(bars4):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontsize=24, family='serif')
        
    plt.suptitle('Community Detection Methods Comparison', fontsize=35, family='serif', y=0.995)
    plt.tight_layout()
    plt.savefig('community_methods_comparison.pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print()
# Visualize the network colored by communities
def plot_communities(G, partition):
    print("8.2.1 Community partition visualization")
    
    n_communities = len(set(partition.values()))
    
    cmap = plt.get_cmap('tab20' if n_communities <= 20 else 'hsv')
    if n_communities > 20:
        colors = [cmap(i / n_communities) for i in range(n_communities)]
    else:
        colors = [cmap(i % 20) for i in range(n_communities)]
    
    # Create color mapping for nodes
    node_colors = [colors[partition[node]] for node in G.nodes()]
    
    fig, ax = plt.subplots(figsize=(20, 16))
    pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
    
    # Draw network
    nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax, alpha=0.8)
    
    ax.set_title('Western US Power Grid Communities', fontsize=60, pad=20, family='serif')
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('WUSPG_communities.pdf', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("8.2.2 Community sizes legend")
    fig, ax = plt.subplots(figsize=(12, 12))

    sizes = Counter(partition.values())
    communities_sorted = sorted(sizes.items(), key=lambda x: x[1], reverse=True)

    comm_ids   = [cid for cid, _ in communities_sorted]
    comm_sizes = [sz  for _,  sz in communities_sorted]

    bar_colors = [colors[cid] for cid in comm_ids]

    # Create widely spaced y positions
    step  = 2.5                    
    y_pos = np.arange(len(comm_ids))*step

    # Draw bars centered on those positions
    bars = ax.barh(
        y_pos,
        comm_sizes,
        color=bar_colors,
        edgecolor='black',
        linewidth=1.5,
        height=1.2                 
    )

    # Put ticks at the same positions
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'#{cid}' for cid in comm_ids],
                    fontsize=16, family='serif')

    ax.set_xlabel('Number of Nodes', fontsize=28, family='serif')
    ax.set_title('Community Sizes', fontsize=35, family='serif', pad=15)
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.grid(axis='x', alpha=0.3)
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax*1.08)

    # Also use y_pos for the value labels
    for y, size in zip(y_pos, comm_sizes):
        ax.text(size+2, y, f'{size}', va='center', ha='left', fontsize=16, family='serif')

    # Expand y-limits so first/last bars are fully visible
    ax.set_ylim(-step, y_pos[-1]+step)

    plt.tight_layout()
    plt.savefig('community_sizes_legend.pdf', dpi=300, bbox_inches='tight')
    plt.close()


# SECTION 9: NETWORK VISUALIZATION 
# Represent the network with node negree and betweenness centrality.
def plot_network(G):
    print("SECTION 9: Network Visualization")
    print("-"*70)
    pos = nx.spring_layout(G, seed=42, k=None)
    
    # Metrics
    print("1. Node degree")
    degrees = dict(G.degree())
    node_deg_vals = np.array(list(degrees.values()))

    print("2. Node betweenness")
    node_betw = nx.betweenness_centrality(G, normalized=True)
    node_betw_vals = np.array(list(node_betw.values()))

    # Node size from degree
    if node_deg_vals.max() == node_deg_vals.min():
        node_sizes = np.full(len(node_deg_vals), (20+200)/2.0)
    else:
        node_sizes = 20+(node_deg_vals-node_deg_vals.min())*(200-20)/(node_deg_vals.max()-node_deg_vals.min())

    # Plotting
    print("3. Plot display")
    fig, ax = plt.subplots(figsize=(12, 10))

    # Edges
    edges = list(G.edges())
    ecoll = nx.draw_networkx_edges(
        G,
        pos,
        ax=ax,
        edge_color='gray',
        width=0.7,
        alpha=0.7
    )

    # Nodes
    ncoll = nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=node_sizes,
        node_color=node_betw_vals,
        cmap=plt.cm.get_cmap('viridis'),
        vmin=node_betw_vals.min(),
        vmax=node_betw_vals.max(),
        linewidths=0.0
    )

    ax.set_axis_off()
    ax.set_title("Western US Power Grid", fontsize=32, family='serif', pad=10)

    # Colorbars
    cbar_nodes = plt.colorbar(ncoll, ax=ax, fraction=0.046, pad=0.02)
    cbar_nodes.set_label("Node betweenness centrality", fontsize=24, family='serif', rotation=270, labelpad=20)
    cbar_nodes.ax.tick_params(labelsize=20)
    plt.tight_layout()
    plt.savefig("WUSPG_network.pdf", dpi=300, bbox_inches='tight')
    plt.close()

### Full analysis

In [ ]:
# Runs full analysis
def main():
    G = load_network('USpowergrid.txt')
    basic_results = analyse_basic_descriptors(G)
    analyse_clustering(G)
    analyse_distances(G, basic_results['largest_cc'])
    analyse_degree_statistics(G)
    analyse_centrality(G, basic_results['largest_cc'])
    analyse_null_models(G)
    community_results = analyse_communities(G)
    community_comparison(community_results)
    plot_communities(G, community_results['partition'])
    plot_network(G)
main()

SECTION 1: Loading Network Data
----------------------------------------------------------------------
Successfully loaded network

